# Практикум: ветки, слияние и конфликты

Python, 9 класс (углублённый). Раздел 3.2, пара 2.

Двенадцать задач тремя уровнями:

- **Уровень 1 (задачи 1–6)** — обязательный. По одной идее на задачу.
- **Уровень 2 (задачи 7–10)** — на сочетание идей.
- **Уровень 3 (задачи 11–12)** — со звёздочкой.

Все задачи — про проект «Дневник погоды» с прошлой пары: модуль `weather.py` читает из файла `days.txt`
записи о погоде (дата, температура, что на небе) и считает простую статистику.

**Как работать.** У каждой задачи своя папка. Ячейка задачи начинается с `start(N)`: эта команда готовит
папку задачи заново — файлы, ветки и историю коммитов. Ниже, на месте `# ваш код здесь`, пишите
команды, как в лекции: `git("switch -c ...")`, `git("merge ...")`, `git('commit -am "..."')`. Ещё
пригодятся `read(путь)`, `write(путь, текст)`, `show(путь)`, `run(...)` и `git_output("...")` — она
возвращает вывод команды Git строкой и ничего не печатает. Ниже строки `# ---------- проверка ----------`
— тесты: они смотрят, что получилось в папке и в репозитории. Если все проверки проходят, внизу
печатается «ок».

Если запутались, просто запустите ячейку ещё раз: `start` вернёт всё в исходное состояние. Перед
решением полезно осмотреться: `git("status")`, `git("branch")` и `git("log --oneline --graph --all")`.

Править файлы удобно так: `write("weather.py", read("weather.py").replace("старый текст", "новый"))`.
Сообщения коммитов слияния Git в ноутбуке принимает сам; в терминале добавляйте `--no-edit`.

В задачах 11 и 12 нужно написать функцию. Первую ячейку с инструментами нужно запустить.

In [ ]:
# готовые инструменты для задач и проверок — писать их не нужно, просто запустите ячейку
import contextlib
import os
import re
import shlex
import shutil
import stat
import subprocess
import sys
import tempfile
import textwrap
from datetime import datetime, timedelta, timezone
from pathlib import Path

sys.dont_write_bytecode = True        # чтобы после правки файла Python не взял его старую версию
_BASE = Path(tempfile.gettempdir()) / "py9_18_practice"
_BASE.mkdir(parents=True, exist_ok=True)
ROOT = _BASE / "task"
_START = datetime(2026, 11, 23, 10, 0, tzinfo=timezone(timedelta(hours=3)))
_ANSI = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")
_ENV = {name: value for name, value in os.environ.items()
        if not name.startswith(("GIT_", "PIP_"))
        and name not in ("VIRTUAL_ENV", "FORCE_COLOR", "CLICOLOR_FORCE", "PY_COLORS")}
_ENV.update({
    "PYTHONDONTWRITEBYTECODE": "1", "PYTHONIOENCODING": "utf-8", "PYTHON_COLORS": "0",
    "NO_COLOR": "1", "TERM": "dumb", "LANGUAGE": "en_US:en",
    # общие настройки Git — во временном файле, ваши настоящие настройки не меняются
    "GIT_CONFIG_GLOBAL": str(_BASE / "gitconfig"), "GIT_CONFIG_NOSYSTEM": "1",
    "GIT_CEILING_DIRECTORIES": str(_BASE), "GIT_PAGER": "cat", "PAGER": "cat",
    "GIT_EDITOR": "true", "GIT_TERMINAL_PROMPT": "0",
})
for _key, _value in {"user.name": "Ученик 9С", "user.email": "student@example.com",
                     "init.defaultBranch": "main", "log.decorate": "short", "merge.conflictStyle": "merge",
                     "core.autocrlf": "false", "advice.detachedHead": "false"}.items():
    subprocess.run(["git", "config", "--global", _key, _value], env=_ENV, check=False)


# ---------------------------------------------------------------- файлы и команды

def _rmtree(path):
    def retry(func, name, *_):
        os.chmod(name, stat.S_IWRITE)         # в Windows файлы внутри .git только для чтения
        func(name)
    if sys.version_info >= (3, 12):
        shutil.rmtree(path, onexc=retry)
    else:
        shutil.rmtree(path, onerror=retry)


def _write(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).lstrip("\n"), encoding="utf-8", newline="\n")


def write(path, text):
    """Записывает файл в папке задачи (остальные файлы не трогает)."""
    _write(ROOT / path, text)


def read(path):
    """Текст файла из папки задачи."""
    return (ROOT / path).read_text(encoding="utf-8")


def remove(path):
    """Удаляет файл или папку в папке задачи."""
    target = ROOT / path
    if target.is_dir() and not target.is_symlink():
        _rmtree(target)
    elif target.exists() or target.is_symlink():
        target.unlink()


def show(path, start=1, end=None):
    """Печатает файл из папки задачи с номерами строк; start и end — с какой по какую строку."""
    print(f"----- {path} -----")
    for number, line in enumerate(read(path).splitlines(), start=1):
        if number >= start and (end is None or number <= end):
            print(f"{number:3d}  {line}")


def show_tree(path=".", depth=None):
    """Показывает файлы и папки задачи; depth — сколько уровней. Содержимое .git не показывается."""
    start = ROOT / path
    print(f"{ROOT.name}/" if path == "." else f"{path}/")

    def walk(folder, level):
        for item in sorted(folder.iterdir(), key=lambda item: item.name):
            folder_like = item.is_dir() and not item.is_symlink()
            print("    " * level + item.name + ("/" if item.is_dir() else ""))
            if folder_like and item.name != ".git" and (depth is None or level < depth):
                walk(item, level + 1)

    walk(start, 1)


def _execute(command):
    result = subprocess.run(command, cwd=ROOT, env=_ENV, capture_output=True, text=True,
                            encoding="utf-8", errors="replace", stdin=subprocess.DEVNULL, timeout=60)
    return result.returncode, _ANSI.sub("", result.stdout), _ANSI.sub("", result.stderr).strip()


def run(*args):
    """Запускает в папке задачи  python args...  — тот Python, на котором работает ноутбук."""
    print("$ python", *args)
    code, out, error = _execute([sys.executable, *args])
    print(out, end="")
    if code != 0 and "Traceback" in error:    # от трассировки оставляем последнюю строку
        error = error.splitlines()[-1]
    if error:
        print("[ошибка]" if code != 0 else "", error)


def _git(args, env=None, cwd=None):
    result = subprocess.run(["git", "-c", "core.quotePath=false", "-c", "color.ui=never", *args],
                            cwd=cwd or ROOT, env=env or _ENV, capture_output=True,
                            stdin=subprocess.DEVNULL, timeout=60)
    result.stdout = result.stdout.decode("utf-8", errors="replace").replace("\r\n", "\n")
    result.stderr = result.stderr.decode("utf-8", errors="replace").replace("\r\n", "\n")
    return result


def _dated_env():
    # время коммита зависит только от их числа: так история задачи одинакова у всех
    count = _git(["rev-list", "--count", "HEAD"]).stdout.strip()
    moment = _START + timedelta(minutes=5 * (int(count) if count.isdigit() else 0))
    stamp = f"{int(moment.timestamp())} +0300"
    return {**_ENV, "GIT_AUTHOR_DATE": stamp, "GIT_COMMITTER_DATE": stamp}


def _clean(text):
    # убираем цвета и строки прогресса, которые Git перерисовывает через \r
    lines = []
    for line in _ANSI.sub("", text).split("\n"):
        if "\r" in line:
            line = line.split("\r")[-1] or line.split("\r")[0]
            if not line.strip():
                continue
        lines.append(line.rstrip())
    return "\n".join(lines).strip("\n")


def git(command):
    """Выполняет в папке задачи команду  git command  и печатает, что она вывела."""
    print("$ git", command)
    result = subprocess.run(["git", "-c", "core.quotePath=false", "-c", "color.ui=never",
                             *shlex.split(command)], cwd=ROOT, env=_dated_env(),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                            timeout=60)
    text = _clean(result.stdout.decode("utf-8", errors="replace").replace("\r\n", "\n"))
    if text:
        print(text)


def git_output(command):
    """Выполняет  git command  и возвращает то, что она напечатала, строкой (ничего не печатает)."""
    result = _git(shlex.split(command), _dated_env())
    if result.returncode != 0:
        raise RuntimeError(f"git {command}: {result.stderr.strip()}")
    return result.stdout


# ---------------------------------------------------------------- для проверок

def log(ref="HEAD"):
    """История ветки от старых коммитов к новым: список пар (хеш, сообщение)."""
    result = _git(["log", "--reverse", "--format=%H%x09%s", ref])
    if result.returncode != 0:
        return []
    return [tuple(line.split("\t", 1)) for line in result.stdout.splitlines()]


def messages(ref="HEAD"):
    """Сообщения коммитов ветки от старых к новым."""
    return [message for _, message in log(ref)]


def changed(commit):
    """Множество файлов, которые изменил коммит."""
    out = _git(["diff-tree", "--no-commit-id", "--name-only", "-r", "--root", commit]).stdout
    return set(out.split("\n")) - {""}


def file_at(commit, path):
    """Текст файла в коммите или None, если файла там нет."""
    result = _git(["show", f"{commit}:{path}"])
    return result.stdout if result.returncode == 0 else None


def status():
    """Короткий git status: список строк вида ' M файл', '?? файл', 'UU файл'."""
    return [line for line in _git(["status", "--porcelain"]).stdout.split("\n") if line]


def branches():
    """Множество имён веток."""
    out = _git(["for-each-ref", "--format=%(refname:short)", "refs/heads"]).stdout
    return set(out.split("\n")) - {""}


def current_branch():
    """Имя текущей ветки."""
    return _git(["branch", "--show-current"]).stdout.strip()


def tip(ref):
    """Хеш коммита, на который указывает ветка (или None, если ветки нет)."""
    result = _git(["rev-parse", "--verify", "--quiet", f"{ref}^{{commit}}"])
    return result.stdout.strip() if result.returncode == 0 else None


def parents(commit="HEAD"):
    """Список родителей коммита (хеши)."""
    return _git(["log", "-1", "--format=%P", commit]).stdout.split()


def merge_commits(ref="HEAD"):
    """Хеши коммитов слияния в истории ветки."""
    return _git(["rev-list", "--merges", ref]).stdout.split()


def is_ancestor(older, newer):
    """Входит ли коммит older в историю newer."""
    return _git(["merge-base", "--is-ancestor", older, newer]).returncode == 0


def merging():
    """Идёт ли сейчас незавершённое слияние."""
    return (ROOT / ".git" / "MERGE_HEAD").exists()


def rebasing():
    """Идёт ли сейчас незавершённый rebase."""
    return (ROOT / ".git" / "rebase-merge").exists() or (ROOT / ".git" / "rebase-apply").exists()


def has_markers(text):
    """Остались ли в тексте маркеры конфликта."""
    return any(line.startswith(("<<<<<<<", "=======", ">>>>>>>")) for line in (text or "").splitlines())


def module_from(text):
    """Выполняет текст модуля и возвращает его переменные словарём."""
    namespace = {"__name__": "checked_module"}
    exec(text, namespace)
    return namespace


def runs_ok(*args):
    """Запускает  python args...  в папке задачи: (код возврата, что напечатано, текст ошибки)."""
    return _execute([sys.executable, *args])


def setup_log(branch="main"):
    """История ветки в том виде, в каком её оставил start: пары (хеш, сообщение)."""
    return _SETUP[ROOT.name]["logs"][branch]


def setup_tip(branch):
    """Хеш, на который указывала ветка сразу после start."""
    return _SETUP[ROOT.name]["tips"][branch]


def prepared(path):
    """Текст файла в рабочей папке в том виде, в каком его оставил start."""
    return _SETUP[ROOT.name]["files"][path]


def git_merge3(base, ours, theirs):
    """Что делает настоящий git merge-file с тремя версиями (списками строк): список строк результата."""
    probe = _BASE / "merge3_probe"
    probe.mkdir(parents=True, exist_ok=True)
    for name, lines in (("base", base), ("ours", ours), ("theirs", theirs)):
        (probe / name).write_text("".join(line + "\n" for line in lines), encoding="utf-8", newline="\n")
    result = _git(["merge-file", "-p", "-L", "ours", "-L", "base", "-L", "theirs", "ours", "base", "theirs"],
                  cwd=probe)
    return result.stdout.splitlines()


@contextlib.contextmanager
def no_merge_base():
    """Внутри блока git merge-base и git show-branch запрещены (для задачи 11)."""
    space = globals()
    saved = space["git"], space["git_output"]

    def guarded(command):
        if re.search(r"\b(merge-base|show-branch)\b", command):
            raise RuntimeError("в этой задаче git merge-base и git show-branch вызывать нельзя")
        return saved[1](command)

    def guarded_print(command):
        guarded(command)
        saved[0](command)

    space["git"], space["git_output"] = guarded_print, guarded
    try:
        yield
    finally:
        space["git"], space["git_output"] = saved


# ---------------------------------------------------------------- проект «Дневник погоды»

_READ_DAYS = '''
def read_days(path):
    """Читает записи: в каждой строке дата, температура и что на небе."""
    days = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        date, temp, sky = line.split()
        days.append((date, int(temp), sky))
    return days
'''

_AVERAGE = '''
def average_temp(days):
    """Средняя температура."""
    total = 0
    for date, temp, sky in days:
        total += temp
    return total / len(days)
'''

_AVERAGE_ROUND = _AVERAGE.replace('"""Средняя температура."""', '"""Средняя температура, до десятых."""').replace(
    "return total / len(days)", "return round(total / len(days), 1)")

_AVERAGE_EMPTY = '''
def average_temp(days):
    """Средняя температура; None, если записей нет."""
    if not days:
        return None
    total = 0
    for date, temp, sky in days:
        total += temp
    return total / len(days)
'''

_RAINY = '''
def rainy_days(days):
    """Даты дождливых дней."""
    return [date for date, temp, sky in days if sky == "дождь"]
'''

_WARMEST = '''
def warmest_day(days):
    """Дата самого тёплого дня."""
    best = days[0]
    for day in days:
        if day[1] > best[1]:
            best = day
    return best[0]
'''

_RANGE = '''
def temp_range(days):
    """Разница между самой высокой и самой низкой температурой."""
    temps = [temp for date, temp, sky in days]
    return max(temps) - min(temps)
'''

_COLD = '''
def cold_days(days, limit=0):
    """Даты дней, когда было не теплее limit градусов."""
    return [date for date, temp, sky in days if temp <= limit]
'''

_UNFINISHED = '''
def coldest_day(days):
    """Дата самого холодного дня."""
    # TODO: дописать, пока возвращает первый день
    return days[0][0]
'''


def _weather(*functions):
    head = '"""Дневник погоды: чтение записей и простая статистика."""\n\nfrom pathlib import Path\n'
    return head + "".join("\n\n" + textwrap.dedent(function).strip("\n") + "\n" for function in functions)


_DAYS_1 = "2026-11-01 6 дождь\n2026-11-02 4 облачно\n2026-11-03 2 снег\n2026-11-04 5 дождь\n2026-11-05 7 солнце\n"
_DAYS_2 = _DAYS_1 + "2026-11-06 3 облачно\n2026-11-07 1 снег\n2026-11-08 4 дождь\n"
_DAYS_3 = _DAYS_2 + "2026-11-09 0 снег\n2026-11-10 -2 солнце\n"
_README = "# Дневник погоды\n\nЗаписи о погоде по дням: дата, температура, что на небе.\n"
_README_USAGE = _README + (
    "\nКак пользоваться:\n\n```python\nfrom weather import read_days, average_temp\n\n"
    "print(average_temp(read_days(\"days.txt\")))\n```\n")
_FIRST = {"weather.py": _weather(_READ_DAYS, _AVERAGE), "days.txt": _DAYS_1, "README.md": _README}
SAMPLE = [(date, int(temp), sky) for date, temp, sky in (line.split() for line in _DAYS_2.splitlines())]

_REPORT = '''\
"""Короткий отчёт по дневнику погоды."""

from weather import average_temp, read_days

days = read_days("days.txt")
print(f"Записей: {len(days)}, средняя температура: {average_temp(days):.1f}")
'''

_PERIOD = "\nЗаписи с 1 по {} ноября.\n"
_D11 = "2026-11-11 1 облачно\n"
_D12 = "2026-11-12 -1 снег\n"
_D13 = "2026-11-13 -3 снег\n"
_D14 = "2026-11-14 0 облачно\n"


def _history_11(variant):
    """Истории для задачи 11: шаги и пары веток, для которых ищем общего предка."""
    def c(message, name):
        return ("commit", message, {f"{name}.txt": message + "\n"})
    if variant == 0:
        steps = [c("c1", "a"), c("c2", "a"), ("new", "feature"), c("f1", "f"), c("f2", "f"),
                 ("switch", "main"), c("c3", "a")]
        pairs = [("main", "feature"), ("feature", "main"), ("main", "main~1"), ("feature~1", "feature")]
    elif variant == 1:
        steps = [c("c1", "a"), c("c2", "a"), ("new", "feature"), c("f1", "f"), ("switch", "main"), c("c3", "a"),
                 ("merge", "feature"), ("switch", "feature"), c("f2", "f"), ("switch", "main"), c("c4", "a")]
        pairs = [("main", "feature"), ("feature", "main"), ("main~1", "feature~1")]
    else:
        steps = [c("c1", "a"), ("new", "side"), c("s1", "s"), ("switch", "main")]
        steps += [c(f"c{n}", "a") for n in range(2, 6)]
        steps += [("new", "long"), c("l1", "l"), c("l2", "l"), c("l3", "l"), ("merge", "side"),
                  ("switch", "main"), c("c6", "a")]
        pairs = [("main", "long"), ("long", "main"), ("side", "long"), ("side", "main")]
    return steps, pairs


_TASKS = {
    1: [("commit", "Первая версия дневника погоды", _FIRST),
        ("commit", "Записи за 6–8 ноября", {"days.txt": _DAYS_2})],
    2: [("commit", "Первая версия дневника погоды", _FIRST),
        ("new", "warmest"),
        ("commit", "Самый тёплый день", {"weather.py": _weather(_READ_DAYS, _AVERAGE, _WARMEST)}),
        ("commit", "Записи за 6–8 ноября", {"days.txt": _DAYS_2}),
        ("switch", "main")],
    3: [("commit", "Первая версия дневника погоды", _FIRST),
        ("new", "readme"),
        ("commit", "Как пользоваться модулем", {"README.md": _README_USAGE}),
        ("switch", "main"),
        ("commit", "Записи за 6–8 ноября", {"days.txt": _DAYS_2})],
    4: [("commit", "Первая версия дневника погоды", {**_FIRST, "weather.py": _weather(_READ_DAYS, _AVERAGE, _RAINY)}),
        ("new", "upper"),
        ("commit", "rainy_days: не зависеть от регистра",
         {"weather.py": _weather(_READ_DAYS, _AVERAGE, _RAINY.replace('sky == "дождь"', 'sky.lower() == "дождь"'))}),
        ("switch", "main"),
        ("commit", "rainy_days: ливень — тоже дождь",
         {"weather.py": _weather(_READ_DAYS, _AVERAGE,
                                 _RAINY.replace('sky == "дождь"', 'sky in ("дождь", "ливень")'))})],
    5: [("commit", "Первая версия дневника погоды", {**_FIRST, "weather.py": _weather(_READ_DAYS, _AVERAGE, _RAINY)}),
        ("new", "experiment"),
        ("commit", "Округлять среднюю", {"weather.py": _weather(_READ_DAYS, _AVERAGE_ROUND, _RAINY)}),
        ("switch", "main"),
        ("commit", "Средняя пустого дневника", {"weather.py": _weather(_READ_DAYS, _AVERAGE_EMPTY, _RAINY)}),
        ("conflict", "experiment")],
    6: [("commit", "Первая версия дневника погоды", _FIRST),
        ("new", "rainy"), ("commit", "Дождливые дни", {"weather.py": _weather(_READ_DAYS, _AVERAGE, _RAINY)}),
        ("switch", "main"), ("merge", "rainy"),
        ("new", "typo"), ("commit", "Опечатка в README", {"README.md": _README.replace("по дням", "по дням.")}),
        ("switch", "main"), ("merge", "typo"),
        ("new", "wind"), ("commit", "Ветер: черновик", {"wind.py": "# здесь будет учёт ветра\n"}),
        ("switch", "main"),
        ("new", "warmest"),
        ("commit", "Самый тёплый день", {"weather.py": _weather(_READ_DAYS, _AVERAGE, _RAINY, _WARMEST)}),
        ("switch", "main"),
        ("commit", "Записи за 6–8 ноября", {"days.txt": _DAYS_2}),
        ("merge", "warmest"),
        ("new", "draft"), ("commit", "Черновик отчёта", {"report_draft.txt": "Ноябрь: холодает.\n"}),
        ("switch", "main")],
    7: [("commit", "Первая версия дневника погоды",
         {**_FIRST, "weather.py": _weather(_READ_DAYS, _AVERAGE, _WARMEST.replace(">", "<"), _RAINY)}),
        ("commit", "Записи за 6–8 ноября", {"days.txt": _DAYS_2}),
        ("new", "cold"),
        ("commit", "Холодные дни",
         {"weather.py": _weather(_READ_DAYS, _AVERAGE, _WARMEST.replace(">", "<"), _RAINY, _COLD)}),
        ("write", {"weather.py": _weather(_READ_DAYS, _AVERAGE, _WARMEST.replace(">", "<"), _RAINY, _COLD,
                                          _UNFINISHED)})],
    8: [("commit", "Первая версия дневника погоды",
         {**_FIRST, "days.txt": _DAYS_3, "README.md": _README + _PERIOD.format(10)}),
        ("new", "week3"),
        ("commit", "Записи за 11–14 ноября",
         {"days.txt": _DAYS_3 + _D11 + _D13 + _D14, "README.md": _README + _PERIOD.format(14)}),
        ("switch", "main"),
        ("commit", "Записи за 11–12 ноября",
         {"days.txt": _DAYS_3 + _D11 + _D12, "README.md": _README + _PERIOD.format(12)})],
    9: [("commit", "Первая версия дневника погоды", {**_FIRST, "days.txt": _DAYS_2}),
        ("new", "rename"),
        ("commit", "Переименовать read_days в load_days",
         {"weather.py": _weather(_READ_DAYS.replace("def read_days", "def load_days"), _AVERAGE)}),
        ("switch", "main"),
        ("commit", "Короткий отчёт", {"report.py": _REPORT})],
    10: [("commit", "Первая версия дневника погоды", _FIRST),
         ("new", "stats"),
         ("commit", "Размах температур", {"weather.py": _weather(_READ_DAYS, _AVERAGE, _RANGE)}),
         ("commit", "Самый тёплый день", {"weather.py": _weather(_READ_DAYS, _AVERAGE, _RANGE, _WARMEST)}),
         ("switch", "main"),
         ("commit", "Записи за 6–8 ноября", {"days.txt": _DAYS_2}),
         ("commit", "Как пользоваться модулем", {"README.md": _README_USAGE})],
}


def _quiet_git(*args, dated=False, may_fail=False):
    result = _git(list(args), _dated_env() if dated else _ENV)
    if result.returncode != 0 and not may_fail:
        raise RuntimeError(f"git {' '.join(args)}: {result.stderr.strip() or result.stdout.strip()}")


def _put(files):
    for name, text in files.items():
        if text is None:
            remove(name)
        else:
            _write(ROOT / name, text)


def start(task, variant=0, quiet=False):
    """Готовит папку задачи task заново: файлы и историю Git."""
    global ROOT
    ROOT = _BASE / f"task_{task}"
    if ROOT.exists():
        _rmtree(ROOT)
    ROOT.mkdir(parents=True)
    steps = _history_11(variant)[0] if task == 11 else _TASKS.get(task, [])
    if steps:
        _quiet_git("init", "-q", "-b", "main")
    for kind, *rest in steps:
        if kind == "commit":
            message, files = rest
            _put(files)
            _quiet_git("add", "-A")
            _quiet_git("commit", "-q", "-m", message, dated=True)
        elif kind == "new":
            _quiet_git("switch", "-q", "-c", rest[0])
        elif kind == "switch":
            _quiet_git("switch", "-q", rest[0])
        elif kind == "merge":
            _quiet_git("merge", "-q", "--no-edit", rest[0], dated=True)
        elif kind == "conflict":
            _quiet_git("merge", "-q", "--no-edit", rest[0], dated=True, may_fail=True)
        elif kind == "write":
            _put(rest[0])
    names = branches()
    _SETUP[ROOT.name] = {
        "logs": {name: log(name) for name in names},
        "tips": {name: tip(name) for name in names},
        "files": {path.relative_to(ROOT).as_posix(): path.read_text(encoding="utf-8")
                  for path in ROOT.rglob("*")
                  if path.is_file() and ".git" not in path.relative_to(ROOT).parts},
    }
    if not quiet:
        print(f"— папка задачи {task} готова —")


_SETUP = {}

if shutil.which("git") is None:
    print("Внимание: Git не найден. Установите его с https://git-scm.com — без него задачи не работают.")
print("инструменты загружены")

## Уровень 1. Обязательный

### Задача 1. Ветка для новой функции

В дневнике два коммита в `main`. Нужна функция `rainy_days` — она уже написана, её текст лежит
в переменной `RAINY`. Пока функция не проверена, её место в отдельной ветке. Создайте ветку `rainy`,
перейдите на неё, допишите `RAINY` в конец `weather.py` и сделайте коммит с сообщением
`Дождливые дни`. Ветка `main` меняться не должна.

*Подсказка:* блоки 1–2 лекции. Создать ветку и сразу перейти на неё можно одной командой.
Дописать текст в конец файла: `write("weather.py", read("weather.py") + RAINY)`.

In [ ]:
start(1)
RAINY = '''

def rainy_days(days):
    """Даты дождливых дней."""
    return [date for date, temp, sky in days if sky == "дождь"]
'''
git("switch -c rainy")
write("weather.py", read("weather.py") + RAINY)
git("add weather.py")
git('commit -m "Дождливые дни"')


# ---------- проверка ----------
assert branches() == {"main", "rainy"}, f"должны быть ветки main и rainy, а есть: {sorted(branches())}"
assert tip("main") == setup_tip("main"), "ветка main не должна меняться: коммит нужно делать в rainy"
history = log("rainy")
assert history[:-1] == setup_log("main"), "ветка rainy должна расти из последнего коммита main"
assert len(history) == len(setup_log("main")) + 1, f"в ветке rainy нужен ровно один новый коммит, а их {len(history) - len(setup_log('main'))}"
assert history[-1][1] == "Дождливые дни", f"сообщение коммита: {history[-1][1]!r}"
assert changed("rainy") == {"weather.py"}, f"коммит должен менять только weather.py, а меняет: {sorted(changed('rainy'))}"
weather = module_from(file_at("rainy", "weather.py"))
assert weather["rainy_days"](SAMPLE) == ["2026-11-01", "2026-11-04", "2026-11-08"], "rainy_days в коммите работает неправильно"
assert status() == [], f"всё должно быть закоммичено, а git status показывает: {status()}"
print("Задача 1: ок")

### Задача 2. Перемотка

В ветке `warmest` два коммита: функция `warmest_day` и новые записи. Пока шла эта работа, в `main`
ничего не менялось. Влейте `warmest` в `main` так, чтобы коммита слияния не появилось, и удалите
ветку `warmest` — она больше не нужна.

*Подсказка:* блок 3 лекции. Если ветка, в которую сливают, не двигалась, Git делает перемотку сам.
Удалять ветку, на которой стоишь, нельзя.

In [ ]:
start(2)
git("merge warmest")
git("branch -d warmest")

# ---------- проверка ----------
assert current_branch() == "main", f"в конце нужно стоять на main, а вы на {current_branch()!r}"
assert tip("main") == setup_tip("warmest"), "main должна указывать на последний коммит бывшей ветки warmest"
assert merge_commits("main") == [], "коммита слияния быть не должно: здесь нужна перемотка (fast-forward)"
assert branches() == {"main"}, f"ветку warmest нужно удалить, а сейчас есть: {sorted(branches())}"
assert status() == [], f"git status должен быть пустым, а показывает: {status()}"
print("Задача 2: ок")

### Задача 3. Коммит слияния

В ветке `readme` в описание добавлен раздел «Как пользоваться». Тем временем в `main` появились новые
записи. Влейте `readme` в `main`. Должен получиться коммит слияния со стандартным сообщением
`Merge branch 'readme'`, в `main` должны оказаться обе работы.

*Подсказка:* блок 3 лекции. Сообщение Git напишет сам, его не нужно задавать.

In [ ]:
start(3)
git("merge --no-edit readme")


# ---------- проверка ----------
assert current_branch() == "main", f"сливать нужно в main, а вы на {current_branch()!r}"
assert parents("main") == [setup_tip("main"), setup_tip("readme")], "последний коммит main должен быть слиянием: первый родитель — прежний main, второй — readme"
assert messages("main")[-1] == "Merge branch 'readme'", f"сообщение коммита слияния: {messages('main')[-1]!r}"
assert "Как пользоваться" in file_at("main", "README.md"), "правка из readme не попала в main"
assert file_at("main", "days.txt").count("\n") == 8, "новые записи из main потерялись"
assert status() == [], f"git status должен быть пустым, а показывает: {status()}"
print("Задача 3: ок")

### Задача 4. Первый конфликт

Функцию `rainy_days` улучшали двое. В ветке `upper` её научили не обращать внимания на регистр:
теперь подходит и `Дождь`. В `main` решили, что `ливень` — тоже дождь. Обе правки задели одну строку.

Влейте `upper` в `main` и разрешите конфликт так, чтобы работали **обе** правки: подходят `дождь`
и `ливень` в любом регистре. Завершите слияние коммитом.

*Подсказка:* блок 4 лекции. Найти строку с маркерами поможет `show("weather.py")`. Замените весь блок
от `<<<<<<<` до `>>>>>>>` одной правильной строкой, например через `read(...).replace(...)` или
записав новый текст функции. Потом `git add` и `git commit --no-edit`.

In [ ]:
start(4)
git("merge upper")
text = read("weather.py")
before = text.split("<<<<<<< HEAD")[0]
after = text.split(">>>>>>> upper")[1]
middle = '    return [date for date, temp, sky in days if sky.lower() in ("дождь", "ливень")]\n'
write("weather.py", before + middle + after.lstrip("\n"))
git("add weather.py")
git("commit --no-edit")


# ---------- проверка ----------
assert not merging(), "слияние не завершено: после исправления нужны git add и git commit"
assert parents("main") == [setup_tip("main"), setup_tip("upper")], "последний коммит main должен быть слиянием main и upper"
text = file_at("main", "weather.py")
assert not has_markers(text), "в weather.py остались маркеры конфликта"
weather = module_from(text)
tests = [("01", 5, "Дождь"), ("02", 3, "ливень"), ("03", 1, "снег"), ("04", 2, "ЛИВЕНЬ"), ("05", 2, "дождь")]
answer = weather["rainy_days"](tests)
assert answer == ["01", "02", "04", "05"], f"rainy_days должна находить дождь и ливень в любом регистре, а вернула {answer}"
assert weather["average_temp"](SAMPLE) == 4.0, "average_temp сломалась"
assert status() == [], f"git status должен быть пустым, а показывает: {status()}"
print("Задача 4: ок")

### Задача 5. Передумали сливать

Вася начал вливать в `main` ветку `experiment`, получил конфликт в `weather.py` и растерялся: файл
весь в маркерах, `git status` пишет непонятное. Верните всё как было до слияния: `main` на прежнем
коммите, рабочая папка чистая, никаких маркеров. Ветку `experiment` не удаляйте, к ней вернутся позже.

*Подсказка:* посмотрите `git status` — он сам подсказывает нужную команду. Блок 4 лекции, таблица
в конце.

In [ ]:
start(5)
git("merge --abort")


# ---------- проверка ----------
assert not merging(), "слияние всё ещё идёт: его нужно отменить"
assert tip("main") == setup_tip("main"), "main должна остаться на прежнем коммите"
assert status() == [], f"рабочая папка должна быть чистой, а git status показывает: {status()}"
assert read("weather.py") == file_at("main", "weather.py"), "weather.py должен совпадать с последним коммитом main"
assert tip("experiment") == setup_tip("experiment"), "ветку experiment нужно оставить как есть"
print("Задача 5: ок")

### Задача 6. Уборка веток

В репозитории накопились ветки. Часть из них уже влита в `main`, а часть — ещё нет. Удалите все ветки,
которые полностью влиты в `main`, а невлитые оставьте. `main` и невлитые ветки меняться не должны.

*Подсказка:* блок 3 лекции. Какие ветки влиты, покажет `git branch --merged`. Удаляйте ключом `-d`:
он сам откажется удалять ветку, в которой есть невлитые коммиты. Ключом `-D` в этой задаче не
пользуйтесь.

In [ ]:
start(6)
merged = git_output("branch --merged main").splitlines()
for b in merged:
    name = b.strip().replace("* ", "")
    if name != "main" and name != "":
        git(f"branch -d {name}")


# ---------- проверка ----------
assert branches() == {"main", "wind", "draft"}, f"должны остаться main и невлитые ветки, а остались: {sorted(branches())}"
assert tip("main") == setup_tip("main"), "main не должна меняться"
assert tip("wind") == setup_tip("wind") and tip("draft") == setup_tip("draft"), "невлитые ветки не должны меняться"
print("Задача 6: ок")

## Уровень 2. На сочетание идей

### Задача 7. Срочное исправление

Вы работаете в ветке `cold`: в ней уже есть коммит с функцией `cold_days`, а в рабочей папке —
незаконченная функция `coldest_day`, которую коммитить рано. Тут выясняется, что в `main` сломана
`warmest_day`: в сравнении стоит `<` вместо `>`, и функция находит самый холодный день.

Нужно:

1. исправить `warmest_day` в `main` коммитом с сообщением `Исправить warmest_day` (только `weather.py`);
2. получить это исправление и в ветке `cold`;
3. вернуться к работе: стоять на `cold`, незаконченная `coldest_day` снова в рабочей папке и по-прежнему
   не закоммичена.

*Подсказка:* блок 2 лекции. Переключиться на `main` с правками в `weather.py` Git не даст: файл в ветках
разный. Уберите правки в тайник, а в конце достаньте. Исправление из `main` в `cold` переносится обычным
`merge`. Найти `<` удобно так: `show("weather.py", 20, 30)`.

In [ ]:
start(7)
git("stash")
git("switch main")
text = read("weather.py")
write("weather.py", text.replace("if day[1] < best[1]:", "if day[1] > best[1]:"))
git("add weather.py")
git('commit -m "Исправить warmest_day"')
git("switch cold")
git("merge --no-edit main")
git("stash pop")


# ---------- проверка ----------
main_log = log("main")
assert main_log[:-1] == setup_log("main"), "в main должен добавиться ровно один коммит поверх прежних"
assert main_log[-1][1] == "Исправить warmest_day", f"сообщение коммита в main: {main_log[-1][1]!r}"
assert changed("main") == {"weather.py"}, f"исправление должно менять только weather.py, а меняет: {sorted(changed('main'))}"
fixed = module_from(file_at("main", "weather.py"))
assert fixed["warmest_day"](SAMPLE) == "2026-11-05", "в main warmest_day всё ещё работает неправильно"
assert "coldest_day" not in file_at("main", "weather.py"), "незаконченная coldest_day не должна попасть в main"
assert is_ancestor("main", "cold"), "исправление из main должно оказаться в ветке cold"
assert "Холодные дни" in messages("cold"), "коммит «Холодные дни» из ветки cold потерялся"
cold = module_from(file_at("cold", "weather.py"))
assert cold["warmest_day"](SAMPLE) == "2026-11-05" and "cold_days" in cold, "в ветке cold должны быть и исправление, и cold_days"
assert "coldest_day" not in file_at("cold", "weather.py"), "coldest_day коммитить рано — она должна остаться только в рабочей папке"
assert current_branch() == "cold", f"в конце нужно стоять на ветке cold, а вы на {current_branch()!r}"
assert "TODO: дописать" in read("weather.py"), "незаконченная coldest_day пропала из рабочей папки"
assert status() in ([" M weather.py"], ["M  weather.py"]), f"в рабочей папке должна быть только правка weather.py, а git status показывает: {status()}"
print("Задача 7: ок")

### Задача 8. Конфликт в двух файлах

Дневник вели двое. В ветке `week3` записаны дни с 11 по 14 ноября, в `main` — 11 и 12 ноября; запись
за 11-е у обоих одинаковая. Обе ветки поменяли и строку в `README.md` о том, за какой период есть
записи.

Влейте `week3` в `main`. Будут конфликты в двух файлах. Разрешите их так:

- в `days.txt` — все записи из обеих веток, каждая один раз, по порядку дат;
- в `README.md` — `Записи с 1 по 14 ноября.`

*Подсказка:* `git status` покажет все файлы с конфликтами. Разрешить нужно каждый, потом `git add`
обоих и один `git commit --no-edit`. Посмотрите, что Git уже слил сам: запись за 11-е одинаковая у обеих
веток, и в конфликт она не попадёт.

In [ ]:
start(8)
git("merge week3")

text = read("days.txt")
lines = []
for line in text.splitlines():
    if line.startswith("<<<<<<<") or line.startswith("=======") or line.startswith(">>>>>>>"):
        continue
    if line.strip() == "":
        continue
    lines.append(line)
lines = sorted(set(lines))
write("days.txt", "\n".join(lines) + "\n")

readme = "# Дневник погоды\n\nЗаписи о погоде по дням: дата, температура, что на небе.\n\nЗаписи с 1 по 14 ноября.\n"
write("README.md", readme)

git("add days.txt README.md")
git("commit --no-edit")


# ---------- проверка ----------
assert not merging(), "слияние не завершено: после исправления нужны git add и git commit"
assert parents("main") == [setup_tip("main"), setup_tip("week3")], "последний коммит main должен быть слиянием main и week3"
days = file_at("main", "days.txt")
readme = file_at("main", "README.md")
assert not has_markers(days) and not has_markers(readme), "в файлах остались маркеры конфликта"
expected = sorted(set(file_at(setup_tip("main"), "days.txt").splitlines()) | set(file_at(setup_tip("week3"), "days.txt").splitlines()))
assert days.splitlines() == expected, f"в days.txt должны быть все записи по одному разу, по порядку дат, а сейчас:\n{days}"
assert "Записи с 1 по 14 ноября." in readme and "по 12 ноября" not in readme, f"в README.md нужен период с 1 по 14 ноября, а там:\n{readme}"
assert status() == [], f"git status должен быть пустым, а показывает: {status()}"
print("Задача 8: ок")

### Задача 9. Слилось, но сломалось

В ветке `rename` функцию `read_days` переименовали в `load_days`. В `main` тем временем появилась
программа `report.py`, которая печатает короткий отчёт и вызывает `read_days`. Влейте `rename` в `main`,
убедитесь, что `report.py` сломался, и почините его отдельным коммитом с сообщением
`report.py: новое имя load_days`. Старое имя `read_days` в проекте больше не должно встречаться —
без «костылей» вроде `read_days = load_days`.

*Подсказка:* блок 5 лекции. Конфликта не будет, а ошибку покажет `run("report.py")`.

In [ ]:
start(9)
git("merge --no-edit rename")
text = read("report.py")
write("report.py", text.replace("read_days", "load_days"))
git("add report.py")
git('commit -m "report.py: новое имя load_days"')


# ---------- проверка ----------
assert is_ancestor(setup_tip("rename"), "main") and is_ancestor(setup_tip("main"), "main"), "в main должны оказаться обе ветки"
assert merge_commits("main"), "ветку rename нужно влить слиянием (коммит слияния)"
assert messages("main")[-1] == "report.py: новое имя load_days", f"последний коммит: {messages('main')[-1]!r}"
weather, report = file_at("main", "weather.py"), file_at("main", "report.py")
assert "def load_days" in weather and "read_days" not in weather, "в weather.py должна быть load_days и не должно быть read_days"
assert "read_days" not in report, "в report.py осталось старое имя read_days"
assert status() == [], f"git status должен быть пустым, а показывает: {status()}"
code, out, error = runs_ok("report.py")
assert code == 0, f"report.py падает: {error.splitlines()[-1] if error else ''}"
assert out.strip() == "Записей: 8, средняя температура: 4.0", f"report.py печатает: {out.strip()!r}"
print("Задача 9: ок")

### Задача 10. Ровная история

В ветке `stats` два коммита: размах температур и самый тёплый день. В `main` тем временем тоже два
коммита. Нужно, чтобы история `main` получилась прямой, без коммита слияния: сначала коммиты `main`,
потом коммиты `stats`. Коммиты, которые уже были в `main`, должны остаться теми же (с теми же хешами).

*Подсказка:* блок 6 лекции. Сначала перенесите `stats` поверх `main`, потом перемотайте `main`.

In [ ]:
start(10)
git("switch stats")
git("rebase main")
git("switch main")
git("merge stats")


# ---------- проверка ----------
assert not rebasing(), "rebase не завершён"
assert merge_commits("main") == [], "в истории main не должно быть коммитов слияния"
main_log = log("main")
assert main_log[:len(setup_log("main"))] == setup_log("main"), "коммиты, которые были в main, должны остаться теми же"
assert [message for _, message in main_log[len(setup_log("main")):]] == ["Размах температур", "Самый тёплый день"], f"после коммитов main должны идти коммиты stats, а история: {messages('main')}"
assert not is_ancestor(setup_tip("stats"), "main"), "коммиты stats нужно перенести (rebase), а не влить"
weather = module_from(file_at("main", "weather.py"))
assert weather["temp_range"](SAMPLE) == 6 and weather["warmest_day"](SAMPLE) == "2026-11-05", "функции из stats потерялись"
assert "Как пользоваться" in file_at("main", "README.md") and file_at("main", "days.txt").count("\n") == 8, "правки из main потерялись"
assert current_branch() == "main" and status() == [], "в конце стойте на main с чистой рабочей папкой"
print("Задача 10: ок")

## Уровень 3. Со звёздочкой

### Задача 11. Общий предок ★

Напишите функцию `merge_base(a, b)`, которая находит **общего предка** двух коммитов так же, как
`git merge-base`: возвращает полный хеш. `a` и `b` — имена веток или выражения вроде `main~1`.

Общий предок — коммит, который входит в историю и `a`, и `b` (сам коммит тоже считается частью своей
истории). Таких обычно много: если `C` — общий предок, то и все его предки — тоже. Нужен **лучший**:
тот, который не является предком другого общего предка. В историях задачи он всегда один.

Команды `git merge-base` и `git show-branch` вызывать нельзя. Проверка сравнит ответы с настоящим
`git merge-base` на трёх историях; в одной из них ближайший по числу шагов общий предок — не лучший.

*Подсказка:* хеш по имени — `git_output(f"rev-parse {a}").strip()`. Все коммиты с родителями —
`git_output("rev-list --parents --all")`: в каждой строке сначала коммит, потом его родители.
Соберите словарь «коммит → родители», найдите множества предков `a` и `b` обходом графа, пересеките
их и выберите из пересечения тот коммит, который не входит в число предков других.

In [ ]:
start(11)
# посмотреть историю: git("log --oneline --graph --all")


def merge_base(a: str, b: str) -> str:
    """Хеш лучшего общего предка коммитов a и b."""
    # ваш код здесь


# ---------- проверка ----------
for variant in range(3):
    start(11, variant, quiet=True)
    for a, b in _history_11(variant)[1]:
        expected = _git(["merge-base", a, b]).stdout.strip()
        with no_merge_base():
            answer = merge_base(a, b)
        assert answer == expected, f"история {variant}, merge_base({a!r}, {b!r}): ответ {answer!r}, а должен быть {expected!r}"
print("Задача 11: ок")

### Задача 12. Слияние трёх версий ★

Напишите функцию `merge3(base, ours, theirs)`, которая сливает три версии файла так, как это делает
Git при трёхстороннем слиянии. Версии даны списками строк одинаковой длины: строки только меняли,
не добавляли и не удаляли, и `i`-я строка всех трёх версий — одно и то же место файла.

Правила:

1. Строка **изменена стороной**, если в этой версии она не такая, как в `base`.
2. **Участок** — непрерывный кусок строк, в каждой из которых хотя бы одна сторона что-то изменила.
   Строки, которые никто не трогал, участки разделяют и попадают в результат как есть.
3. Участок целиком решается так:
   - если `ours` и `theirs` на участке совпадают — берём их;
   - если одна сторона на участке ничего не меняла (совпадает с `base`) — берём другую;
   - иначе — **конфликт**: строка `<<<<<<< ours`, строки `ours` участка, строка `=======`, строки
     `theirs` участка, строка `>>>>>>> theirs`.

Функция возвращает список строк результата. Проверка сравнит ответы с настоящим `git merge-file`
на примерах, где между участками не меньше четырёх нетронутых строк, а в конфликтах у сторон
различаются первая и последняя строка — на таких примерах Git действует ровно по этим правилам.

*Подсказка:* идите по индексу `i` и, встретив изменённую строку, найдите конец участка вторым циклом.
Срезы `ours[i:j]`, `theirs[i:j]`, `base[i:j]` удобно сравнивать целиком.

In [ ]:
def merge3(base: list[str], ours: list[str], theirs: list[str]) -> list[str]:
    """Трёхстороннее слияние версий одинаковой длины по правилам из условия."""
    # ваш код здесь


# ---------- проверка ----------
B = ["def average_temp(days):", '    """Средняя температура."""', "    total = 0", "    for date, temp, sky in days:",
     "        total += temp", "    return total / len(days)", "", "", "def rainy_days(days):",
     '    """Даты дождливых дней."""', '    return [date for date, temp, sky in days if sky == "дождь"]']


def edit(**changes):
    """Копия B, в которой строки с номерами из changes заменены: edit(l5="...")."""
    lines = B[:]
    for key, value in changes.items():
        lines[int(key[1:])] = value
    return lines


cases = [
    (B, B, B),
    (B, edit(l5="    return round(total / len(days), 1)"), B),
    (B, B, edit(l10='    return [date for date, temp, sky in days if sky in ("дождь", "ливень")]')),
    (B, edit(l1='    """Средняя."""'), edit(l10='    return [date for date, temp, sky in days if sky.lower() == "дождь"]')),
    (B, edit(l5="    return total / max(len(days), 1)"), edit(l5="    return total / max(len(days), 1)")),
    (B, edit(l5="    return round(total / len(days), 1)"), edit(l5="    return round(total / len(days))")),
    (B, edit(l4="        total = total + temp"), edit(l5="    return total // len(days)")),
    (B, edit(l1='    """Средняя за все дни."""', l2="    total = 0.0", l9='    """Дни с дождём."""'),
        edit(l1='    """Средняя по дням."""', l2="    total: float = 0", l10="    return []")),
    (B, edit(l2="    total = 0.0", l3="    for _, temp, _ in days:"), edit(l3="    for day in days:")),
    ([], [], []),
]
for base, ours, theirs in cases:
    saved = (base[:], ours[:], theirs[:])
    answer = merge3(base, ours, theirs)
    assert (base, ours, theirs) == saved, "функция не должна менять переданные списки"
    expected = git_merge3(base, ours, theirs)          # спрашиваем настоящий Git
    assert answer == expected, "ответ не совпал с git merge-file.\nДолжно быть:\n" + "\n".join(expected) + "\n\nА получилось:\n" + "\n".join(map(str, answer or []))
print("Задача 12: ок")

## Что дальше

На следующей паре — то же самое, но вдвоём и через GitHub: удалённый репозиторий, `push` и `pull`,
запросы на слияние и обзор кода. Нужен аккаунт на [github.com](https://github.com). Заранее
потренироваться можно во вкладке «Удалённые репозитории» в
[Learn Git Branching](https://learngitbranching.js.org/?locale=ru_RU).